In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# --- PASSO 1: DIVISÃO DOS DADOS ---
X_reg, y_reg = make_regression(n_samples=1000, n_features=8, noise=15.0, random_state=42)

X_train_full, X_test, y_train_full, y_test = train_test_split(X_reg, y_reg, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

X_train_full_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

def get_reg_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "R2": round(r2_score(y_true, y_pred), 4),
        "MSE": round(mse, 4),
        "RMSE": round(np.sqrt(mse), 4),
        "MAE": round(mean_absolute_error(y_true, y_pred), 4),
        "MAPE": round(mean_absolute_percentage_error(y_true, y_pred), 4)
    }

models_reg = {
    "Linear Regression": {
        "class": LinearRegression,
        "params": [{}]
    },
    "Decision Tree Regressor": {
        "class": DecisionTreeRegressor,
        "params": [{"max_depth": d, "random_state": 42} for d in [3, 5, 10]]
    },
    "Random Forest Regressor": {
        "class": RandomForestRegressor,
        "params": [{"n_estimators": n, "max_depth": 5, "random_state": 42} for n in [20, 50]]
    },
    "Linear Regression Lasso": {
        "class": Lasso,
        "params": [{"alpha": a, "random_state": 42} for a in [0.01, 0.1, 1.0]]
    },
    "Linear Regression Ridge": {
        "class": Ridge,
        "params": [{"alpha": a, "random_state": 42} for a in [0.1, 1.0, 10.0]]
    },
    "Linear Regression Elastic Net": {
        "class": ElasticNet,
        "params": [{"alpha": a, "l1_ratio": 0.5, "random_state": 42} for a in [0.01, 0.1]]
    },
    "Polinomial Regression": {
        "pipeline": lambda d, **kw: Pipeline([("poly", PolynomialFeatures(degree=d)), ("reg", LinearRegression(**kw))]),
        "params": [{"degree": d} for d in [2, 3]]
    },
    "Polinomial Regression Lasso": {
        "pipeline": lambda d, **kw: Pipeline([("poly", PolynomialFeatures(degree=d)), ("reg", Lasso(random_state=42, **kw))]),
        "params": [{"degree": 2, "alpha": 0.1}, {"degree": 2, "alpha": 1.0}]
    },
    "Polinomial Regression Ridge": {
        "pipeline": lambda d, **kw: Pipeline([("poly", PolynomialFeatures(degree=d)), ("reg", Ridge(random_state=42, **kw))]),
        "params": [{"degree": 2, "alpha": 0.1}, {"degree": 2, "alpha": 1.0}]
    },
    "Polinomial Regression Elastic Net": {
        "pipeline": lambda d, **kw: Pipeline([("poly", PolynomialFeatures(degree=d)), ("reg", ElasticNet(random_state=42, **kw))]),
        "params": [{"degree": 2, "alpha": 0.1, "l1_ratio": 0.5}, {"degree": 2, "alpha": 1.0, "l1_ratio": 0.5}]
    }
}

res_train_r, res_val_r, res_test_r = [], [], []

for name, config in models_reg.items():
    # --- PASSOS 2, 3 e 4: MODELO DEFAULT ---
    if "pipeline" in config:
        default_model = config["pipeline"](2) # grau 2 como padrão para polinomial
    else:
        default_model = config["class"]()
        
    default_model.fit(X_train_scaled, y_train)
    
    # Passo 3: Treino (Default)
    res_train_r.append({"Algoritmo": name, **get_reg_metrics(y_train, default_model.predict(X_train_scaled))})
    
    # Passo 4: Validação (Default)
    res_val_r.append({"Algoritmo": name, **get_reg_metrics(y_val, default_model.predict(X_val_scaled))})
    
    # --- PASSO 5: BUSCA PELO MELHOR HIPERPARÂMETRO ---
    best_score = -float('inf')
    best_p = None
    
    for p in config["params"]:
        params_copy = p.copy()
        if "pipeline" in config:
            deg = params_copy.pop("degree")
            model = config["pipeline"](deg, **params_copy)
        else:
            model = config["class"](**params_copy)
            
        model.fit(X_train_scaled, y_train)
        y_val_pred = model.predict(X_val_scaled)
        score = r2_score(y_val, y_val_pred)
        
        if score > best_score:
            best_score = score
            best_p = p.copy()
            
    # --- PASSOS 6 e 7: RETREINAMENTO (TREINO + VALIDAÇÃO) ---
    p_copy = best_p.copy()
    if "pipeline" in config:
        deg = p_copy.pop("degree")
        final_model = config["pipeline"](deg, **p_copy)
    else:
        final_model = config["class"](**p_copy)
        
    final_model.fit(X_train_full_scaled, y_train_full)
    
    # --- PASSO 8: PERFORMANCE NO TESTE ---
    res_test_r.append({"Algoritmo": name, **get_reg_metrics(y_test, final_model.predict(X_test_scaled))})

# Exibição das Tabelas do Ensaio
print("=== 1) REGRESSÃO: DADOS DE TREINO (DEFAULT) ===")
print(pd.DataFrame(res_train_r).to_string(index=False))

print("\n=== 2) REGRESSÃO: DADOS DE VALIDAÇÃO (DEFAULT) ===")
print(pd.DataFrame(res_val_r).to_string(index=False))

print("\n=== 3) REGRESSÃO: DADOS DE TESTE (MODELO OTIMIZADO) ===")
print(pd.DataFrame(res_test_r).to_string(index=False))

=== 1) REGRESSÃO: DADOS DE TREINO (DEFAULT) ===
                        Algoritmo     R2       MSE    RMSE     MAE   MAPE
                Linear Regression 0.9923  233.5939 15.2838 12.1274 0.4687
          Decision Tree Regressor 1.0000    0.0000  0.0000  0.0000 0.0000
          Random Forest Regressor 0.9720  853.1062 29.2080 22.5330 0.7218
          Linear Regression Lasso 0.9920  242.2688 15.5650 12.3583 0.4412
          Linear Regression Ridge 0.9923  233.6836 15.2867 12.1356 0.4672
    Linear Regression Elastic Net 0.8742 3833.1228 61.9122 49.7725 0.5339
            Polinomial Regression 0.9928  218.9624 14.7974 11.6990 0.3575
      Polinomial Regression Lasso 0.9922  236.6247 15.3826 12.1946 0.4103
      Polinomial Regression Ridge 0.9928  219.0574 14.8006 11.6998 0.3566
Polinomial Regression Elastic Net 0.8746 3819.8141 61.8046 49.6611 0.5208

=== 2) REGRESSÃO: DADOS DE VALIDAÇÃO (DEFAULT) ===
                        Algoritmo     R2        MSE     RMSE     MAE   MAPE
          